# ALE3DCNN Best Recipe + Global Context Variants

Run All does the full staged pipeline:

1. Run the current best plain CNN recipe using the original improved plain-CNN autoencoder checkpoint.
2. For each ResNet/global-context variant, train a matching autoencoder first.
3. Use that matching autoencoder checkpoint to initialize the ResNet/global-context contrastive run.

This avoids loading plain-CNN weights into ResNet encoders while still preserving the winning recipe: autoencoder-pretrained CNN encoder, InfoNCE fine-tuning, pretrained trainable text projection.


In [ ]:
import os, sys, time, json, platform
from pathlib import Path
print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', exc)


In [ ]:
!pip -q install nilearn nibabel huggingface-hub safetensors adapters transformers pyarrow matplotlib pandas scikit-learn tqdm umap-learn
REPO_URL = os.environ.get('NEUROVLM_REPO_URL', 'https://github.com/neurovlm/neurovlm.git')
REPO_BRANCH = os.environ.get('NEUROVLM_REPO_BRANCH', 'neurovlm_gnn')
REPO_DIR = os.environ.get('NEUROVLM_REPO_DIR', '/content/neurovlm_gnn')
if not os.path.exists(REPO_DIR):
    !git clone --branch $REPO_BRANCH --single-branch $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin $REPO_BRANCH
    !git -C $REPO_DIR checkout $REPO_BRANCH
    !git -C $REPO_DIR pull --ff-only origin $REPO_BRANCH
os.chdir(REPO_DIR)
!pip -q install -e '.[viz,notebook,metrics]'
sys.path.insert(0, str(Path(REPO_DIR) / 'src'))
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

DRIVE_ROOT = '/content/drive/MyDrive/neurovlm'
RUNS_DIR = f'{DRIVE_ROOT}/runs_ale_3dcnn_best_global'
CACHE_DIR = f'{DRIVE_ROOT}/data_ale_3dcnn/ale_caches'
LOCAL_CACHE_DIR = '/content/ale_caches_ale_3dcnn'
EVAL_RESOURCE_DIR = '/content/drive/MyDrive/neurovlm_evaluation_resources'
RUN_STAMP = time.strftime('%Y%m%d_%H%M%S')
for d in [RUNS_DIR, CACHE_DIR, LOCAL_CACHE_DIR, EVAL_RESOURCE_DIR]:
    os.makedirs(d, exist_ok=True)

def find_autoencoder_checkpoint():
    env_path = os.environ.get('ALE_CNN_AE_CHECKPOINT', '')
    if env_path and Path(env_path).exists():
        return str(Path(env_path))
    exact_best = Path(DRIVE_ROOT) / 'runs_ale_3dcnn_autoencoder_pretrain' / 'autoencoder_atlas_free_20260510_194641' / 'checkpoints' / 'best_cnn_autoencoder.pt'
    if exact_best.exists():
        return str(exact_best)
    candidates = []
    roots = [
        Path(DRIVE_ROOT) / 'runs_ale_3dcnn_autoencoder_pretrain',
        Path(DRIVE_ROOT) / 'runs_ale_3dcnn_full_pipeline',
        Path(DRIVE_ROOT) / 'runs_ale_3dcnn_best_global',
        Path('runs_ale_3dcnn_autoencoder_pretrain'),
    ]
    best_patterns = [
        'autoencoder_atlas_free_*/checkpoints/best_cnn_autoencoder.pt',
        'stage1_autoencoder_atlas_free_*/checkpoints/best_cnn_autoencoder.pt',
        '**/checkpoints/best_cnn_autoencoder.pt',
        '**/checkpoints/best_generation_top5_dice.pt',
    ]
    fallback_patterns = [
        'autoencoder_atlas_free_*/checkpoints/last_cnn_autoencoder.pt',
        'stage1_autoencoder_atlas_free_*/checkpoints/last_cnn_autoencoder.pt',
        '**/checkpoints/last_cnn_autoencoder.pt',
    ]
    for patterns in [best_patterns, fallback_patterns]:
        candidates = []
        for root in roots:
            if root.exists():
                for pattern in patterns:
                    candidates.extend(root.glob(pattern))
        existing = [p for p in candidates if p.exists()]
        if existing:
            return str(max(existing, key=lambda p: p.stat().st_mtime))
    return ''

AUTOENCODER_CHECKPOINT = find_autoencoder_checkpoint()
CACHE_FILE = f'{LOCAL_CACHE_DIR}/atlas_free_ale_4mm_fwhm9p0_crop_float16.pt'
COMPARISON_FILE = f'{RUNS_DIR}/best_global_context_comparison.csv'

AE_EPOCHS = int(os.environ.get('ALE_CNN_AE_EPOCHS', '150'))
CONTRASTIVE_EPOCHS = int(os.environ.get('ALE_CNN_CONTRASTIVE_EPOCHS', '200'))
SKIP_COMPLETED = os.environ.get('ALE_CNN_SKIP_COMPLETED', '1') != '0'
print('AUTOENCODER_CHECKPOINT:', AUTOENCODER_CHECKPOINT or '<not found yet>')
if not AUTOENCODER_CHECKPOINT:
    print('Set ALE_CNN_AE_CHECKPOINT or AUTOENCODER_CHECKPOINT to your Drive checkpoint before run_variant().')


In [ ]:
def base_args(autoencoder_checkpoint, epochs=None):
    return [
        '--mode', 'atlas_free',
        '--epochs', str(CONTRASTIVE_EPOCHS if epochs is None else epochs),
        '--batch-size-auto',
        '--lr-cnn', '1e-4', '--lr-proj', '1e-5',
        '--warmup-epochs', '5', '--temperature', '0.07',
        '--val-interval', '5', '--early-stopping-patience', '25',
        '--out-dim', '384', '--dropout', '0.1', '--norm', 'group',
        '--kernel-fwhm-mm', '9', '--resolution-mm', '4', '--cache-dtype', 'float16',
        '--cache-file', CACHE_FILE,
        '--encoder-init', 'autoencoder_pretrained', '--autoencoder-checkpoint', autoencoder_checkpoint,
        '--text-proj-init', 'pretrained_infonce',
        '--semantic-eval', '--eval-resource-dir', EVAL_RESOURCE_DIR,
        '--train-sanity-n', '512', '--num-workers', '4',
        '--comparison-file', COMPARISON_FILE,
    ]


def autoencoder_args(v, run_dir):
    args = [
        '--mode', 'atlas_free',
        '--model', v['model'],
        '--epochs', str(v.get('ae_epochs', AE_EPOCHS)),
        '--batch-size-auto', '--batch-size-candidates', v.get('ae_batch_size_candidates', '64,32,16,8,4'),
        '--lr', str(v.get('ae_lr', 3e-4)), '--weight-decay', '1e-4',
        '--val-interval', '5', '--early-stopping-patience', '20',
        '--base-channels', str(v['base_channels']),
        '--num-blocks', str(v['num_blocks']),
        '--blocks-per-stage', str(v.get('blocks_per_stage', 2)),
        '--latent-dim', '384', '--dropout', '0.1', '--norm', 'group',
        '--kernel-fwhm-mm', '9', '--resolution-mm', '4', '--cache-dtype', 'float16',
        '--cache-file', CACHE_FILE,
        '--lambda-recon', '1.0', '--lambda-dice', '0.5', '--lambda-topk', '0.5', '--lambda-corr', '0.25',
        '--recon-alpha', '10.0', '--recon-gamma', '1.0',
        '--device', 'cuda' if torch.cuda.is_available() else 'auto',
        '--num-workers', '4',
        '--run-dir', str(run_dir), '--checkpoint-dir', str(Path(run_dir) / 'checkpoints'),
    ]
    if '--use-dilation' in v.get('extra', []):
        args.append('--use-dilation')
    if '--multi-scale' in v.get('extra', []):
        args.append('--multi-scale')
    if '--global-context' in v.get('extra', []):
        idx = v['extra'].index('--global-context')
        args.extend(['--global-context', v['extra'][idx + 1]])
    return args


PLAIN_BEST_VARIANTS = [
    dict(name='ae_pretrained_finetune_cnn_pretrained_text_trainable_plain', model='ale_3dcnn', base_channels=48, num_blocks=4, extra=[]),
]

# Backwards-compatible alias used by older cells. This intentionally contains only the runnable plain best recipe.
VARIANTS = PLAIN_BEST_VARIANTS

GLOBAL_CONTEXT_VARIANTS = [
    dict(name='ae_pretrained_finetune_resnet48_multiscale_attention', model='ale_3dcnn_resnet', base_channels=48, num_blocks=4, extra=['--multi-scale', '--global-context', 'attention']),
    dict(name='ae_pretrained_finetune_resnet64_multiscale_attention', model='ale_3dcnn_resnet', base_channels=64, num_blocks=4, extra=['--multi-scale', '--global-context', 'attention'], ae_batch_size_candidates='32,16,8,4'),
    dict(name='ae_pretrained_finetune_resnet48_dilated_multiscale_attention', model='ale_3dcnn_resnet', base_channels=48, num_blocks=4, extra=['--use-dilation', '--multi-scale', '--global-context', 'attention']),
    dict(name='ae_pretrained_finetune_resnet48_5stage_multiscale_se', model='ale_3dcnn_resnet', base_channels=48, num_blocks=5, extra=['--multi-scale', '--global-context', 'se'], ae_batch_size_candidates='32,16,8,4'),
]


def run_command(cmd, done_file=None):
    if done_file is not None and SKIP_COMPLETED and Path(done_file).exists():
        print('Skipping completed:', done_file)
        return 0
    print(cmd)
    return os.system(cmd)


def run_variant(v):
    autoencoder_checkpoint = v.get('autoencoder_checkpoint') or AUTOENCODER_CHECKPOINT or find_autoencoder_checkpoint()
    if not autoencoder_checkpoint:
        raise ValueError('Could not find an autoencoder checkpoint. Set AUTOENCODER_CHECKPOINT or ALE_CNN_AE_CHECKPOINT.')
    run_dir = Path(RUNS_DIR) / f'{v["name"]}_{RUN_STAMP}'
    args = base_args(autoencoder_checkpoint) + [
        '--model', v['model'],
        '--base-channels', str(v['base_channels']),
        '--num-blocks', str(v['num_blocks']),
        '--run-dir', str(run_dir),
        '--checkpoint-dir', str(run_dir / 'checkpoints'),
    ] + v.get('extra', [])
    cmd = 'python experiments/train_ale_cnn.py ' + ' '.join(map(str, args))
    return run_command(cmd, done_file=run_dir / 'eval_results.json')


def pretrain_autoencoder_for_variant(v):
    if v.get('autoencoder_checkpoint') and Path(v['autoencoder_checkpoint']).exists():
        print('Using preset autoencoder checkpoint:', v['autoencoder_checkpoint'])
        return v['autoencoder_checkpoint']
    run_dir = Path(RUNS_DIR) / f'autoencoder_{v["name"]}_{RUN_STAMP}'
    best_ckpt = run_dir / 'checkpoints' / 'best_cnn_autoencoder.pt'
    if not best_ckpt.exists():
        args = autoencoder_args(v, run_dir)
        cmd = 'python experiments/train_ale_cnn_autoencoder.py ' + ' '.join(map(str, args))
        code = run_command(cmd, done_file=best_ckpt)
        if code != 0:
            raise RuntimeError(f'Autoencoder pretraining failed: {v["name"]}')
    v['autoencoder_checkpoint'] = str(best_ckpt)
    print('Autoencoder checkpoint ready:', best_ckpt)
    return str(best_ckpt)


def run_global_context_variant(v):
    pretrain_autoencoder_for_variant(v)
    code = run_variant(v)
    if code != 0:
        raise RuntimeError(f'Contrastive variant failed: {v["name"]}')
    return code


In [ ]:
# Stage 1: run the plain best recipe using the original improved plain-CNN autoencoder checkpoint.
for variant in PLAIN_BEST_VARIANTS:
    code = run_variant(variant)
    if code != 0:
        raise RuntimeError(f'Variant failed: {variant["name"]}')


In [ ]:
# Stage 2: for each global-context variant, pretrain the matching autoencoder first,
# then run the contrastive fine-tuning recipe with that matching checkpoint.
for variant in GLOBAL_CONTEXT_VARIANTS:
    run_global_context_variant(variant)
